In [ ]:
# ============================================================
# CONFIGURATION - every path comes from config/paths.py, the single
# source of truth. Override cluster locations with the MUSICA_ENV_*
# environment variables documented there. Do not hard-code paths here.
# ============================================================
import sys, pathlib
_here = pathlib.Path.cwd().resolve()
_ROOT = next(p for p in [_here, *_here.parents]
             if (p / 'config' / 'paths.py').exists())
sys.path.insert(0, str(_ROOT))
import config  # also puts functions/ on sys.path
from config import paths as P


This script uses MUSICA ne30 runs and tries to 
- Merge the surface hourly dataset into one .nc file for a given range of dates
- Loop through all monitors 
For each monitor
1. Extract the closest cell using the ColIdx (need to get that first)
2. Get the surface hourly O3 simulations 
3. Convert the hourly O3 to local time given the lat-lon of the monitor
4. For a given range of dates, calculate the daily MDA8 O3 for each day => get a timeseries at this location

*Also save hourly surface O3 in UTC for their reference*

In [ ]:
Output_diri = f'{P.PROCESSED_DATA_ROOT}/CONUSBGO3_postprocessing/'
MonitorInfo_filepath = f'{Output_diri}MonitorInfo/given_monitors.csv'
Monitorne30Idx_filepath = f'{Output_diri}MonitorInfo/MatchedMonitors_ne30_ColIdx.csv'

In [ ]:
# a diri to place all related files for this project
svante_archive = f'{P.ARCHIVE}/'

# list for all case names
BASE2022_casename = 'f.e22.FCnudged.ne30_ne30_mg17.BGO3.BASEY20220401TY20230401'
BASE2023_casename = 'f.e22.FCnudged.ne30_ne30_mg17.BGO3.BASEY20230401TY20231101'
noBB2022_casename = 'f.e22.FCnudged.ne30_ne30_mg17.BGO3.noBBemisCONUS80kmBufferY20220401TY20221101'
noBB2023_casename = 'f.e22.FCnudged.ne30_ne30_mg17.BGO3.noBBemisCONUS80kmBufferY20230401TY20231101'
noAnthro2022_casename = 'f.e22.FCnudged.ne30_ne30_mg17.BGO3.noANTHROemisCONUS80kmBufferY20220401TY20221101'
noAnthro2023_casename = 'f.e22.FCnudged.ne30_ne30_mg17.BGO3.noANTHROemisCONUS80kmBufferY20230401TY20231101'

fileregion_ls = ['WestCoast','Mountain','Midwest','Southwest','Northeast','Southeast']
nfileregion = len(fileregion_ls)

# label for each casename
casename_label_dic = {
                'SLAMS':'SLAMS',
                BASE2022_casename:'BASE2022',
                BASE2023_casename:'BASE2023',
                ### Perturbations
                noBB2022_casename:'noBB2022',
                noBB2023_casename:'noBB2023',
                noAnthro2022_casename:'noAnthro2022',
                noAnthro2023_casename:'noAnthro2023',
                }


In [ ]:
# merged h2 surface file for each case
mergedh2_surfO3filepath_dic = {
                                BASE2022_casename:f'{Output_diri}h2_surfO3_merged/f.e22.FCnudged.ne30_ne30_mg17.BGO3.BASEY20220401TY20230401.cam.h2.surflev.O3.2022-04-01T2022-11-01.nc',
                                BASE2023_casename:f'{Output_diri}h2_surfO3_merged/f.e22.FCnudged.ne30_ne30_mg17.BGO3.BASEY20230401TY20231101.cam.h2.surflev.O3.2023-04-01T2023-10-31.nc',
                                ### Perturbations
                                noBB2022_casename:f'{Output_diri}h2_surfO3_merged/f.e22.FCnudged.ne30_ne30_mg17.BGO3.noBBemisCONUS80kmBufferY20220401TY20221101.cam.h2.surflev.O3.2022-04-01T2022-10-31.nc',
                                noBB2023_casename:f'{Output_diri}h2_surfO3_merged/f.e22.FCnudged.ne30_ne30_mg17.BGO3.noBBemisCONUS80kmBufferY20230401TY20231101.cam.h2.surflev.O3.2023-04-01T2023-11-01.nc',
                                noAnthro2022_casename:f'{Output_diri}h2_surfO3_merged/f.e22.FCnudged.ne30_ne30_mg17.BGO3.noANTHROemisCONUS80kmBufferY20220401TY20221101.cam.h2.surflev.O3.2022-04-01T2022-11-01.nc',
                                noAnthro2023_casename:f'{Output_diri}h2_surfO3_merged/f.e22.FCnudged.ne30_ne30_mg17.BGO3.noANTHROemisCONUS80kmBufferY20230401TY20231101.cam.h2.surflev.O3.2023-04-01T2023-11-01.nc',
                                }

In [ ]:
mergedh2_surfO3filepath_dic[BASE2022_casename]

#### Functions and Setup

In [ ]:
import sys
# functions I defined
sys.path.insert(0,f'{P.HOME_ROOT}/Scripts/CESM_analysis/functions/')
from Plot_2D import Plot_2D # To draw a map
from func_MUSICA_DefineRegion import *

SCRIP_ne30 = f'{P.HOME_ROOT}/Scripts/CESM_analysis/functions/ne30np4_091226_pentagons.nc'

In [ ]:
import os
import glob
import fnmatch

import pandas as pd
import geopandas as gpd
from shapely import wkt
from shapely.geometry import Point, Polygon

import xarray as xr
import numpy as np

import matplotlib.pyplot as plt # Core library for plotting
import matplotlib.cm as cm # To use different colormaps
import cartopy.crs as ccrs # For map projection
import seaborn as sns # boxplot

from matplotlib.cm import ScalarMappable


In [ ]:
# my functions
import sys
sys.path.append(f'{P.HOME_ROOT}/Scripts/CESM_analysis/functions')
from func_ModelEval_statistical_tests import *
from SE_analysis import get_site_index

In [ ]:
MonitorIdx_df = pd.read_csv(Monitorne30Idx_filepath)

In [ ]:
from timezonefinder import TimezoneFinder
from datetime import datetime
import pytz

def get_summer_offset(lati: float, loni: float) -> int:
    # Find the timezone name from lat/lon
    tf = TimezoneFinder()
    tz_name = tf.timezone_at(lat=lati, lng=loni)
    if tz_name is None:
        raise ValueError("Could not determine timezone for given coordinates")

    # Pick a midsummer date (July 1) to ensure DST applies if relevant
    dt = datetime(datetime.now().year, 7, 1)

    # Localize to that timezone
    tz = pytz.timezone(tz_name)
    localized_dt = tz.localize(dt, is_dst=True)

    # Get offset from UTC in hours
    offset_hours = int(localized_dt.utcoffset().total_seconds() // 3600)
    return offset_hours

# Example
lati, loni = 33.545278, -86.549167
print(get_summer_offset(lati, loni))  # → -5


In [ ]:
def extract_date_correctly(filename):
    # Extracting the date portion from filename based on observed structure
    date_section = filename.split('.')[-2].split('-')
    # Combine the first three elements to form the date in YYYY-MM-DD
    full_date = '-'.join(date_section[:3])
    return full_date


In [ ]:
# Specify timezone for the given region
def regional_UTCtimezone_offset_summer(fileregion):
    """This function returns the timezone of a given region
       *With daylight saving  
       Options: WestCoast,Mountain,Midwest,Southwest,Southeast,Northeast
    """
    # Define regions        
    if fileregion == "WestCoast":
        # Pacific (UTC-7)
        UTCtimezone_offset = -7
    elif fileregion == "Mountain":
        # Mountain (UTC-6)
        UTCtimezone_offset = -6
    elif fileregion == "Midwest":
        # Central (UTC-5)
        UTCtimezone_offset = -5
    elif fileregion == "Southwest":
        # Central (UTC-5)
        UTCtimezone_offset = -5
    elif fileregion == "Southeast":
        # Eastern (UTC-4)
        UTCtimezone_offset = -4
    elif fileregion == "Northeast":
        # Eastern (UTC-4)
        UTCtimezone_offset = -4
    
    return UTCtimezone_offset

### Extract the ColIdx for all the given monitors
Processed in GetMatched_ne30_GivenMonitors_ColumnIndex.py

In [ ]:
import pandas as pd
from pathlib import Path

def load_monitor_info(MonitorInfo_filepath) -> pd.DataFrame:
    """Load monitor metadata with columns: lon, lat, site_name, AQS_code."""
    path = Path(MonitorInfo_filepath)
    usecols = ["lon", "lat", "site_name", "AQS_code"]
    dtypes  = {"site_name": "string", "AQS_code": "string"}  # keep leading zeros

    if path.suffix.lower() in {".xlsx", ".xls"}:
        df = pd.read_excel(path, usecols=usecols, dtype=dtypes)
    else:
        # sep=None lets pandas infer comma/space/tab; engine='python' required for that
        df = pd.read_csv(path, usecols=usecols, dtype=dtypes, sep=None, engine="python")

    # Basic cleaning
    df.columns = [c.strip() for c in df.columns]
    df["site_name"] = df["site_name"].str.strip()
    df["AQS_code"]  = df["AQS_code"].str.strip()

    # Ensure numeric coords
    df["lat"] = pd.to_numeric(df["lat"], errors="coerce")
    df["lon"] = pd.to_numeric(df["lon"], errors="coerce")

    # Convert any 0–360 longitudes to −180–180
    needs_wrap = df["lon"] > 180
    if needs_wrap.any():
        df.loc[needs_wrap, "lon"] = df.loc[needs_wrap, "lon"] - 360

    # Drop rows missing coordinates
    df = df.dropna(subset=["lat", "lon"]).reset_index(drop=True)

    # Add a stable site index (0..N-1) you can carry into NetCDF as the 'site' dimension
    df.insert(0, "site", range(len(df)))

    return df

# Usage
MonitorInfo_df = load_monitor_info(MonitorInfo_filepath)


In [ ]:
MonitorInfo_df.head(5)

In [ ]:
# Keep only the unique MonitorID values along with their Latitude and Longitude
unique_monitor_locations = MonitorInfo_df[['AQS_code', 'lat', 'lon']].drop_duplicates().reset_index(drop=True)


In [ ]:
# Get an example output file
ex_h2_filepath = f'{P.ARCHIVE}/f.e22.FCnudged.ne30_ne30_mg17.BGO3.BASEY20220401TY20230401/atm/hist/f.e22.FCnudged.ne30_ne30_mg17.BGO3.BASEY20220401TY20230401.cam.h2.2022-05-04-03600.nc'
ds_ne30 = xr.open_dataset(ex_h2_filepath).isel(lev=lev_idx,ilev=lev_idx,time=10)

In [ ]:
lev_idx

In [ ]:
unique_monitor_locations.AQS_code.values[:1]

In [ ]:

# loop through the monitors and find the corresponding column index
ls_Index_MonitorIDi = []
ls_Model_lati = []
ls_Model_loni = []

# try one monitor or several
for MonitorIDi in unique_monitor_locations.AQS_code.values[:5]:
# # if all
# for MonitorIDi in unique_monitor_locations.AQS_code.values:
    # Get the latitude and longitude values for 'MonitorIDi'
    MonitorIDi_df = unique_monitor_locations[unique_monitor_locations['AQS_code'] == MonitorIDi]

    lati = MonitorIDi_df['lat'].iloc[0]
    loni = MonitorIDi_df['lon'].iloc[0]

    # find the model index
    Index_MonitorIDi = get_site_index( site_lat=lati, site_lon=360+loni, scrip_file=SCRIP_ne30 )
    if Index_MonitorIDi==None:
        # add to the list
        ls_Index_MonitorIDi.append('Find None')
        ls_Model_lati.append(Model_lati)
        ls_Model_loni.append(Model_loni)

    else:
        Model_lati = ds_ne30.lat.values[Index_MonitorIDi]
        Model_loni = ds_ne30.lon.values[Index_MonitorIDi]
        # add to the list
        ls_Index_MonitorIDi.append(Index_MonitorIDi)
        ls_Model_lati.append(Model_lati)
        ls_Model_loni.append(Model_loni)
        
# append to the df
unique_monitor_locations['MUSICA0_colIndex'] = ls_Index_MonitorIDi
unique_monitor_locations['Approx_MUSICA0_lat'] = ls_Model_lati
unique_monitor_locations['Approx_MUSICA0_lon'] = ls_Model_loni

# # Save the DataFrame to a CSV file
# unique_monitor_locations.to_csv(Savefile_path, index=False)

# print("Saved to:",Savefile_path)

In [ ]:
ls_Model_lati

In [ ]:
ds_ne30.lat.values[Index_MonitorIDi],ds_ne30.lon.values[Index_MonitorIDi]

In [ ]:
ls_Index_MonitorIDi

In [ ]:
lati,loni,ls_Index_MonitorIDi,ls_Model_lati,ls_Model_loni

### Merge the surface hourly O3 for a given range of dates into one .nc file
Processed in Merge_h2files_hourlysurfO3.py

In [ ]:
# merged file
casename = BASE2022_casename
YYYY = casename_label_dic[casename][-4:]
startfileDate = f'{YYYY}-04-01'
endfileDate = f'{YYYY}-11-01'
HourlyFilePath = f'{Output_diri}h2_surfO3_merged/{casename}.cam.h2.surflev.{startfileDate}T{endfileDate}.nc'
HourlyO3_da = xr.open_dataset(HourlyFilePath)

In [ ]:
# read in 
casename = BASE2022_casename
histfreq = 'h2'
RunPath = f'{svante_archive}{casename}/atm/hist/'
RunFiles = sorted(glob.glob(os.path.join(RunPath, f'*{histfreq}*')))

In [ ]:
import os
import re
import glob

casename  = BASE2022_casename
histfreq  = "h2"
startMMDD = "0401"
endMMDD   = "1101"

RunPath = f"{svante_archive}{casename}/atm/hist/"

date_re = re.compile(r"\.(\d{4})-(\d{2})-(\d{2})-\d+\.nc$")  # ...YYYY-MM-DD-XXXXX.nc

def extract_mmdd(fname: str) -> str:
    m = date_re.search(fname)
    if not m:
        return None
    # Return 'MMDD'
    return f"{m.group(2)}{m.group(3)}"

def mmdd_in_range(mmdd: str, start_mmdd: str, end_mmdd: str) -> bool:
    """Inclusive range check on MMDD; supports wrap-around (e.g., Nov–Mar)."""
    if mmdd is None:
        return False
    if start_mmdd <= end_mmdd:
        return start_mmdd <= mmdd <= end_mmdd
    else:
        # wrap-around across year end (e.g., start='1101', end='0201')
        return (mmdd >= start_mmdd) or (mmdd <= end_mmdd)

# Gather and filter
all_files = glob.glob(os.path.join(RunPath, f"*{histfreq}*.nc"))
RunFiles  = sorted(f for f in all_files if mmdd_in_range(extract_mmdd(f), startMMDD, endMMDD))

print(f"Selected {len(RunFiles)} files in MMDD [{startMMDD}–{endMMDD}]")
# Optional: peek at the first/last few
print("\n".join(RunFiles[:3] + (["..."] if len(RunFiles) > 6 else []) + RunFiles[-3:]))


### For one monitor, get the approximated hourly O3, convert time zone, then calculate daily MDA8 for given dates

In [ ]:
# Generate a list of dates
casename = BASE2022_casename
YYYY = casename_label_dic[casename][-4:]
startfileDate = f'{YYYY}-04-01'
endfileDate = f'{YYYY}-10-31'

# Generate daily date range
date_range = pd.date_range(start=startfileDate, end=endfileDate, freq="D")

# Two versions of formatted strings
datesv1 = date_range.strftime("%Y-%m-%d").tolist()  # 'YYYY-MM-DD'
datesv2 = date_range.strftime("%Y%m%d").tolist()    # 'YYYYMMDD'

print(len(datesv1), "dates generated")
print("First few v1:", datesv1[:5])
print("First few v2:", datesv2[:5])

In [ ]:
HourlyFilePath

In [ ]:
# Use merged surface hourly file
HourlyFilePath = mergedh2_surfO3filepath_dic[casename]
HourlyO3_da = xr.open_dataset(HourlyFilePath)['O3']
HourlyO3_da

In [ ]:
# try one monitor or several
for MonitorIDi in MonitorIdx_df.AQS_code.values[:100]:
# # if all
# for MonitorIDi in MonitorIdx_df.AQS_code.values:
    # Get the latitude and longitude values
    MonitorIDi_df = MonitorIdx_df[MonitorIdx_df['AQS_code'] == MonitorIDi]

    lati = MonitorIDi_df['lat'].iloc[0]
    loni = MonitorIDi_df['lon'].iloc[0]
    colIdxi = MonitorIDi_df['MUSICA0_colIndex'].iloc[0]


In [ ]:
lati,loni,colIdxi

In [ ]:
MonitorIDi

In [ ]:
Monitori_HourlyO3_da = HourlyO3_da.sel(ncol=int(colIdxi))
Monitori_HourlyO3_da

Following U.S. EPA convention, 8-hour rolling means were computed with a requirement of at least 6 valid hours per window. Seventeen 8-hour blocks ending between 07:00 and 23:00 each day were considered. The daily MDA8 was taken as the maximum of these blocks, and a day was considered valid only if at least 13 of the 17 blocks met the minimum-hours criterion.

In [ ]:
import pandas as pd
import xarray as xr

def compute_mda8_UTCoffset(
    o3_hourly_utc: xr.DataArray,
    datesv1: list[str],
    utc_offset_hours: int,         # e.g., -5 for CDT
    min_hours_per_block: int = 6,  # ≥6 of 8 hours to form a valid 8-h mean
    min_blocks_per_day: int = 13   # ≥13 of the 17 daily blocks required
) -> xr.DataArray:
    """
    Compute daily MDA8 O3 from hourly O3 timestamps in UTC,
    using a fixed integer local-time offset (hours) for the rolling windows.

    - Shifts the time coordinate by `utc_offset_hours` to a 'local' clock
      (local = UTC + offset), performs rolling and day grouping,
      then returns results indexed by local dates in `datesv1`.
    - Works with extra dims (e.g., 'site') and dask-backed arrays.
    """

    # 1) Shift time axis to 'local' clock (no tz objects, just a simple offset)
    t_local = pd.DatetimeIndex(o3_hourly_utc.time.values) + pd.to_timedelta(utc_offset_hours, "h")
    O3 = o3_hourly_utc.assign_coords(time=("time", t_local))
    
    # 2) 8-hour rolling mean with EPA min-hours criterion
    O3_8h = O3.rolling(time=8, min_periods=min_hours_per_block).mean()
    
    # 3) Keep only the 17 blocks ending 07..23 local
    O3_8h_17 = O3_8h.where(O3_8h["time"].dt.hour.isin(np.arange(7, 24)))

    # 4) Group by local civil day; compute daily max and valid-block count
    day_idx = pd.DatetimeIndex(O3_8h_17.time.values).floor("D")
    O3_8h_17 = O3_8h_17.assign_coords(day=("time", day_idx))
    
    mda8_by_day    = O3_8h_17.groupby("day").max("time", skipna=True)
    blocks_per_day = O3_8h_17.groupby("day").count("time")

    # 5) Apply day validity (≥13 of 17 blocks)
    valid = blocks_per_day >= min_blocks_per_day
    mda8_by_day = mda8_by_day.where(valid)

    # 6) Reindex to requested local dates and return with standard 'time' coord
    target_days = pd.to_datetime(datesv1)  # these are local dates
    out = mda8_by_day.reindex(day=target_days)
    out = out.rename({"day": "time"}).assign_coords(time=("time", target_days))
    return out
    
    # return t_local


In [ ]:
Monitori_HourlyO3_da

In [ ]:
datesv1[-1]

In [ ]:
MDA8O3 = compute_mda8_UTCoffset(Monitori_HourlyO3_da, datesv1, utc_offset_hours=get_summer_offset(lati, loni))

In [ ]:
# MDA8O3

In [ ]:
Monitori_MDA8O3 = MDA8O3
Monitori_MDA8O3

In [ ]:
import matplotlib.pyplot as plt

def plot_monitor_timeseries(Monitori_MDA8O3, MonitorIDi: str):
    """
    Plot daily MDA8 O3 for one monitor with its AQS code in the title.
    
    Parameters
    ----------
    Monitori_MDA8O3 : xr.DataArray
        Daily O3 values for one site (dims: time)
    MonitorIDi : str
        Informative label for the site, e.g., its AQS code
    """
    fig, ax = plt.subplots(figsize=(8, 4))

    # Convert from mol/mol to ppb
    vals_ppb = Monitori_MDA8O3 * 1e9

    vals_ppb.plot(ax=ax, marker="o", linestyle="-", color="tab:blue", markersize=3)

    ax.set_ylabel("MDA8 O$_3$ (ppb)", fontsize=15)
    ax.set_xlabel("Date", fontsize=12)
    ax.set_title(f"MDA8 O$_3$ at Monitor {MonitorIDi}", fontsize=18)
    
    # Add horizontal line at 70 ppb
    ax.axhline(70, color="red", linestyle="--", linewidth=1)
    ax.text(
        x=vals_ppb["time"].values[len(vals_ppb) // 20],  # put near the left edge
        y=72, 
        s="MDA8 O$_3$ = 70 ppb",
        color="red",
        fontsize=15,
        va="bottom",
    )

    plt.tight_layout()
    plt.show()



In [ ]:
# Example call
plot_monitor_timeseries(Monitori_MDA8O3, MonitorIDi)  # replace with your site’s AQS code


##### Other attempts

In [ ]:
import pandas as pd
import xarray as xr

def compute_mda8_UTCoffset(
    o3_hourly_utc: xr.DataArray,
    datesv1: list[str],
    utc_offset_hours: int,         # e.g., -5 for CDT
    min_hours_per_block: int = 6,  # ≥6 of 8 hours to form a valid 8-h mean
    min_blocks_per_day: int = 13   # ≥13 of the 17 daily blocks required
) -> xr.DataArray:
    """
    Compute daily MDA8 O3 from hourly O3 timestamps in UTC,
    using a fixed integer local-time offset (hours) for the rolling windows.

    - Shifts the time coordinate by `utc_offset_hours` to a 'local' clock
      (local = UTC + offset), performs rolling and day grouping,
      then returns results indexed by local dates in `datesv1`.
    - Works with extra dims (e.g., 'site') and dask-backed arrays.
    """

    # 1) Shift time axis to 'local' clock (no tz objects, just a simple offset)
    t_local = pd.DatetimeIndex(o3_hourly_utc.time.values) + pd.to_timedelta(utc_offset_hours, "h")
    O3 = o3_hourly_utc.assign_coords(time=("time", t_local))
    
    # 2) 8-hour rolling mean with EPA min-hours criterion
    O3_8h = O3.rolling(time=8, min_periods=min_hours_per_block).mean()
    
    # 3) Keep only the 17 blocks ending 07..23 local
    O3_8h_17 = O3_8h.where(O3_8h["time"].dt.hour.isin(np.arange(7, 24)))

    # 4) Group by local civil day; compute daily max and valid-block count
    day_idx = pd.DatetimeIndex(O3_8h_17.time.values).floor("D")
    O3_8h_17 = O3_8h_17.assign_coords(day=("time", day_idx))
    
    mda8_by_day    = O3_8h_17.groupby("day").max("time", skipna=True)
    blocks_per_day = O3_8h_17.groupby("day").count("time")

    # 5) Apply day validity (≥13 of 17 blocks)
    valid = blocks_per_day >= min_blocks_per_day
    mda8_by_day = mda8_by_day.where(valid)

    # 6) Reindex to requested local dates and return with standard 'time' coord
    target_days = pd.to_datetime(datesv1)  # these are local dates
    out = mda8_by_day.reindex(day=target_days)
    out = out.rename({"day": "time"}).assign_coords(time=("time", target_days))

    # Metadata
    out = out.assign_attrs({
        "long_name": "Daily maximum of 8-hour average ozone (MDA8)",
        "cell_methods": "time: mean (interval: 8 hours) time: maximum within days",
        "comment": (f"Computed from UTC timestamps using local offset {utc_offset_hours} h "
                    f"(local = UTC + {utc_offset_hours}). 8h rolling mean requires ≥{min_hours_per_block}/8 "
                    f"hours; day valid if ≥{min_blocks_per_day} of 17 end-hours (07–23) present."),
        "time_basis": "local clock constructed via fixed UTC offset",
    })
    return out


In [ ]:
MDA8O3 = compute_mda8_UTCoffset(Monitori_HourlyO3_da['O3'], datesv1, utc_offset_hours=get_summer_offset(lati, loni))

In [ ]:
MDA8O3

In [ ]:
# convert the time to local time based on the lat lon

In [ ]:
# #-----------------
# ### Time zone has to be specific, since it depends on which city!!
# #-----------------
# # Deal with Timezone for local time
# # Original Time
# time_ar = Monitori_HourlyO3_da.time.values

# # Store the LT
# MonitoriLT_dt_ls = []
# MonitoriLT_formatdt_ls = []

# # Convert the time 
# for timeidx in range(len(time_ar)):
#     # Convert numpy.datetime64 object to Python datetime object
#     dt = np.datetime64(time_ar[timeidx])
#     # Convert time zone | To the local time of this city
#     adjustedHours = get_summer_offset(lati, loni)
#     MonitoriLT_dt = np.datetime64(dt - np.timedelta64(-adjustedHours, 'h'))
#     MonitoriLT_dt = MonitoriLT_dt.astype('datetime64[s]').item()
#     MonitoriLT_dt_ls.append(MonitoriLT_dt)

#     # Format the datetime
#     MonitoriLT_formatdt = MonitoriLT_dt.strftime('%Y-%m-%d %H:%M')
#     MonitoriLT_formatdt_ls.append(MonitoriLT_formatdt)

# #-----------------------------------------------------------------------
# # Create a copy of the original DataArray
# MonitoriLT_da = Monitori_da.copy()

# # Check if MonitoriLT_dt_ls has the same number of elements as the 'time' dimension of the DataArray copy
# if len(MonitoriLT_dt_ls) == MonitoriLT_da.sizes['time']:
#     # Assign new time values from MonitoriLT_dt_ls to the copy
#     MonitoriLT_da = MonitoriLT_da.assign_coords(time=MonitoriLT_dt_ls)
# else:
#     print("The new time list does not match the size of the 'time' dimension.")

In [ ]:
# import pandas as pd
# import xarray as xr

# def compute_mda8_local(o3_hourly: xr.DataArray, datesv1: list[str],
#                        min_hours_per_block: int = 6,  # ≥6 of 8 hours to form a valid 8-h mean
#                        min_blocks_per_day: int = 13   # ≥13 of 17 daily blocks to accept the day
#                       ) -> xr.DataArray:
#     """
#     Compute daily MDA8 O3 from hourly local-time ozone. 

#     Parameters
#     ----------
#     o3_hourly : xr.DataArray with a 'time' dimension (can include extra dims like 'site')
#     datesv1   : list[str] of 'YYYY-MM-DD' dates to return (inclusive range you want)
#     """

#     # 8-hour rolling mean (label at window end); require at least min_hours_per_block
#     O3_8h = o3_hourly.rolling(time=8, min_periods=min_hours_per_block).mean()

#     # Keep only the 17 blocks that END 07..23 local each day (EPA convention)
#     end_hours = xr.DataArray(pd.DatetimeIndex(O3_8h.time.values).hour, dims=("time",))
#     O3_8h_17 = O3_8h.where((end_hours >= 7) & (end_hours <= 23))

#     # Group by civil day and compute daily max + count of valid blocks
#     day_idx = pd.DatetimeIndex(O3_8h_17.time.values).floor("D")
#     O3_8h_17 = O3_8h_17.assign_coords(day=("time", day_idx))

#     mda8_by_day   = O3_8h_17.groupby("day").max("time", skipna=True)
#     blocks_per_day = O3_8h_17.groupby("day").count("time")

#     # Apply daily validity: need at least min_blocks_per_day valid 8-h means
#     valid = blocks_per_day >= min_blocks_per_day
#     mda8_by_day = mda8_by_day.where(valid)

#     # Reindex to requested dates, preserve order; rename 'day'->'time'
#     target_days = pd.to_datetime(datesv1)
#     out = mda8_by_day.reindex(day=target_days)
#     out = out.rename({"day": "time"}).assign_coords(time=("time", target_days))

#     # Metadata
#     out = out.assign_attrs({
#         "long_name": "Daily maximum of 8-hour average ozone (MDA8)",
#         "cell_methods": "time: mean (interval: 8 hours) time: maximum within days",
#         "comment": (f"8h rolling mean (>= {min_hours_per_block}/8 hours valid). "
#                     f"Daily value valid if >= {min_blocks_per_day} of 17 end-hours (07–23) present.")
#     })
#     return out

# MDA8O3 = compute_mda8_local(MonitoriLT_da['O3'], datesv1)

## Test

# Get MUSICA simulated surface O3 as daily MDA8 for a 800 sites
Complete script see 'Extract_givenmonitorO3_hourly_dailyMDA8_toNetCDF.py'

In [ ]:
"""Function to calculate MDA8 O3 in the local time following EPA's guidance"""
import pandas as pd
import xarray as xr
def compute_mda8_UTCoffset(
    o3_hourly_utc: xr.DataArray,
    datesv1: list[str],
    utc_offset_hours: int,         # e.g., -5 for CDT
    min_hours_per_block: int = 6,  # ≥6 of 8 hours to form a valid 8-h mean
    min_blocks_per_day: int = 13   # ≥13 of the 17 daily blocks required
) -> xr.DataArray:
    """
    Compute daily MDA8 O3 from hourly O3 timestamps in UTC,
    using a fixed integer local-time offset (hours) for the rolling windows.

    - Shifts the time coordinate by `utc_offset_hours` to a 'local' clock
      (local = UTC + offset), performs rolling and day grouping,
      then returns results indexed by local dates in `datesv1`.
    - Works with extra dims (e.g., 'site') and dask-backed arrays.
    """

    # 1) Shift time axis to 'local' clock (no tz objects, just a simple offset)
    t_local = pd.DatetimeIndex(o3_hourly_utc.time.values) + pd.to_timedelta(utc_offset_hours, "h")
    O3 = o3_hourly_utc.assign_coords(time=("time", t_local))
    
    # 2) 8-hour rolling mean with EPA min-hours criterion
    O3_8h = O3.rolling(time=8, min_periods=min_hours_per_block).mean()
    
    # 3) Keep only the 17 blocks ending 07..23 local
    O3_8h_17 = O3_8h.where(O3_8h["time"].dt.hour.isin(np.arange(7, 24)))

    # 4) Group by local civil day; compute daily max and valid-block count
    day_idx = pd.DatetimeIndex(O3_8h_17.time.values).floor("D")
    O3_8h_17 = O3_8h_17.assign_coords(day=("time", day_idx))
    
    mda8_by_day    = O3_8h_17.groupby("day").max("time", skipna=True)
    blocks_per_day = O3_8h_17.groupby("day").count("time")

    # 5) Apply day validity (≥13 of 17 blocks)
    valid = blocks_per_day >= min_blocks_per_day
    mda8_by_day = mda8_by_day.where(valid)

    # 6) Reindex to requested local dates and return with standard 'time' coord
    target_days = pd.to_datetime(datesv1)  # these are local dates
    out = mda8_by_day.reindex(day=target_days)
    out = out.rename({"day": "time"}).assign_coords(time=("time", target_days))
    return out
    

In [ ]:
# For a given case
casename = BASE2022_casename

startMMDD = '04-01'
endMMDD = '10-31'

In [ ]:

import numpy as np
import pandas as pd
import xarray as xr

def casei_build_and_save_all_sites(casename, MonitorIdx_df, startMMDD, endMMDD, Output_diri):
    """
    Build and save datasets for daily MDA8 O3 (local time) and hourly O3 (UTC)
    at all monitor sites for one case/year.

    Outputs
    -------
    - LocalTimeMDA8O3.<year>.GivenMonitors.<startDate>T<endDate>.nc
    - UTChourlyO3.<year>.GivenMonitors.<startDate>T<endDate>.nc
    """

    #-----------------------------------------
    # Generate an inclusive list of dates for the target year
    YYYY = casename_label_dic[casename][-4:]
    startfileDate = f"{YYYY}-{startMMDD}"
    endfileDate   = f"{YYYY}-{endMMDD}"
    date_range = pd.date_range(start=startfileDate, end=endfileDate, freq="D")
    datesv1 = date_range.strftime("%Y-%m-%d").tolist()

    print(len(datesv1), "dates generated")
    print("First few v1:", datesv1[:5])

    #-----------------------------------------
    # Open merged hourly surface O3 for the case (expected UTC timestamps)
    HourlyFilePath = mergedh2_surfO3filepath_dic[casename]
    HourlyO3_da = xr.open_dataset(HourlyFilePath)["O3"]

    #-----------------------------------------
    # Build lists for concatenation
    mda8_list = []
    hourly_list = []
    tdays = pd.to_datetime(datesv1)

    for row in MonitorIdx_df.itertuples(index=False):
        colIdxi   = int(row.MUSICA0_colIndex)
        loni      = float(row.lon)
        lati      = float(row.lat)
        aqs_codei = str(row.AQS_code)

        # Hourly O3 for this site (UTC)
        Monitori_HourlyO3_da = HourlyO3_da.sel(ncol=colIdxi)

        # Compute MDA8 (local time) for this site
        MDA8O3 = compute_mda8_UTCoffset(
            Monitori_HourlyO3_da, datesv1,
            utc_offset_hours=get_summer_offset(lati, loni)
        )

        # Wrap MDA8 into a Dataset (singleton site)
        ds_mda8 = xr.Dataset(
            data_vars={
                "MDA8O3": (("site", "time"), MDA8O3.values[np.newaxis, :],
                           {"units": "ppb",
                            "long_name": "Daily maximum of 8-hour average ozone (MDA8)",
                            "cell_methods": "time: mean (interval: 8 hours) time: maximum within days"})
            },
            coords={
                "time":     ("time", tdays),
                "lat":      ("site", [lati]),
                "lon":      ("site", [loni]),
                "AQS_code": ("site", [aqs_codei]),
            },
        )
        mda8_list.append(ds_mda8)

        # Wrap Hourly O3 into a Dataset (singleton site)
        ds_hourly = xr.Dataset(
            data_vars={
                "O3": (("site", "time"), Monitori_HourlyO3_da.values[np.newaxis, :],
                       {"units": Monitori_HourlyO3_da.attrs.get("units", "mol/mol"),
                        "long_name": "Hourly surface O3 (UTC)"})
            },
            coords={
                "time":     ("time", Monitori_HourlyO3_da.time.values),
                "lat":      ("site", [lati]),
                "lon":      ("site", [loni]),
                "AQS_code": ("site", [aqs_codei]),
            },
        )
        hourly_list.append(ds_hourly)

    # Concatenate all sites
    ds_all_mda8   = xr.concat(mda8_list, dim="AQS_code")
    ds_all_hourly = xr.concat(hourly_list, dim="AQS_code")

    # Add coordinate metadata
    for ds in [ds_all_mda8, ds_all_hourly]:
        ds["lat"].attrs.update({"units": "degrees_north", "standard_name": "latitude"})
        ds["lon"].attrs.update({"units": "degrees_east",  "standard_name": "longitude"})

    # Global attrs
    ds_all_mda8.attrs.update({
        "title": "Daily MDA8 O3 at point monitors",
        "Conventions": "CF-1.9",
        "comment": "Local-time MDA8 computed from UTC hourly O3 using fixed summertime UTC offset "
                   "(8h rolling means with >=6 valid hours; day valid if >=13 of 17 end-hours 07–23).",
    })
    ds_all_hourly.attrs.update({
        "title": "Hourly O3 at point monitors (UTC)",
        "Conventions": "CF-1.9",
        "comment": "Extracted from merged MUSICA outputs at monitor column indices (UTC timestamps).",
    })

    #-----------------------------------------
    # Save files
    enc_mda8 = {
        "MDA8O3": {"zlib": True, "complevel": 5,
                   "chunksizes": (min(200, ds_all_mda8.dims["site"]), ds_all_mda8.dims["time"])},
        "lat": {"zlib": True, "complevel": 5},
        "lon": {"zlib": True, "complevel": 5},
        "AQS_code": {"zlib": True, "complevel": 5},
    }

    enc_hourly = {
        "O3": {"zlib": True, "complevel": 5,
               "chunksizes": (min(200, ds_all_hourly.dims["site"]), ds_all_hourly.dims["time"])},
        "lat": {"zlib": True, "complevel": 5},
        "lon": {"zlib": True, "complevel": 5},
        "AQS_code": {"zlib": True, "complevel": 5},
    }

    allMonitors_MDA8O3_filename = (
        f"{Output_diri}LocalTimeMDA8O3.{casename_label_dic[casename]}."
        f"GivenMonitors.{startfileDate}T{endfileDate}.nc"
    )
    allMonitors_HourlyO3_filename = (
        f"{Output_diri}UTChourlyO3.{casename_label_dic[casename]}."
        f"GivenMonitors.{startfileDate}T{endfileDate}.nc"
    )

    ds_all_mda8.to_netcdf(allMonitors_MDA8O3_filename, format="NETCDF4", encoding=enc_mda8)
    ds_all_hourly.to_netcdf(allMonitors_HourlyO3_filename, format="NETCDF4", encoding=enc_hourly)

    print(f"Saved MDA8 dataset to   {allMonitors_MDA8O3_filename}")
    print(f"Saved Hourly dataset to {allMonitors_HourlyO3_filename}")


In [ ]:
MonitorIdx_df.head(3)

In [ ]:
# Build once per year, then save
ds_all_sites = build_mda8_for_all_sites(MonitorInfo_df, datesv1)

# Compression & chunking (good defaults)
enc = {
    "MDA8O3": {"zlib": True, "complevel": 5, "chunksizes": (min(200, ds_all_sites.dims["site"]), ds_all_sites.dims["time"])},
    "lat": {"zlib": True, "complevel": 5},
    "lon": {"zlib": True, "complevel": 5},
    "site_name": {"zlib": True, "complevel": 5},
    "AQS_code":  {"zlib": True, "complevel": 5},
    # let xarray write time as CF-compliant by default
}

allMonitors_MDA8O3_filename = f'{Output_diri}LocalTimeMDA8O3.{casename_label_dic[casename]}.GivenMonitors.{startfileDate}T{endfileDate}.nc'
ds_all_sites.to_netcdf(allMonitors_MDA8O3_filename, format="NETCDF4", encoding=enc)
print(f"Save to {allMonitors_MDA8O3_filename}")

In [ ]:
allMonitors_MDA8O3_filename = f'{Output_diri}LocalTimeMDA8O3.{casename_label_dic[casename]}.GivenMonitors.{startfileDate}T{endfileDate}.nc'
allMonitors_MDA8O3_filename

In [ ]:
allMonitors_HourlyO3_filename = f'{Output_diri}UTChourlyO3.{casename_label_dic[casename]}.GivenMonitors.{startfileDate}T{endfileDate}.nc'
allMonitors_HourlyO3_filename

In [ ]:
MonitorIdx_df

In [ ]:
MonitorIdx_df.MUSICA0_colIndex.values[:20]

In [ ]:
# MonitorIdx_df = MonitorIdx_df[pd.to_numeric(MonitorIdx_df["MUSICA0_colIndex"], errors="coerce").notna()]
# MonitorIdx_df["MUSICA0_colIndex"] = MonitorIdx_df["MUSICA0_colIndex"].astype(int)


In [ ]:
# Remove rows where MUSICA0_colIndex is 'Find None' or NaN
MonitorIdx_df = MonitorIdx_df[MonitorIdx_df["MUSICA0_colIndex"] != "Find None"].copy()
MonitorIdx_df = MonitorIdx_df.dropna(subset=["MUSICA0_colIndex"])
# Now cast to int
MonitorIdx_df["MUSICA0_colIndex"] = MonitorIdx_df["MUSICA0_colIndex"].astype(int)


In [ ]:
MonitorIdx_df.MUSICA0_colIndex.values[:20]

# Plot to check the processed output

In [ ]:
import os

def print_file_size(filepath: str) -> None:
    """
    Print the size of a file in human-readable units (B, KB, MB, GB, TB).

    Parameters
    ----------
    filepath : str
        Path to the file whose size you want to check.
    """
    try:
        size_bytes = os.path.getsize(filepath)
    except FileNotFoundError:
        print(f"File not found: {filepath}")
        return

    # Convert to human-readable
    units = ["B", "KB", "MB", "GB", "TB"]
    size = float(size_bytes)
    for unit in units:
        if size < 1024.0:
            print(f"{filepath}\nSize: {size:.2f} {unit}")
            return
        size /= 1024.0
    # If somehow larger than TB
    print(f"{filepath}\nSize: {size:.2f} PB")


In [ ]:
import matplotlib.pyplot as plt

def plot_monitor_timeseries(Monitori_MDA8O3, MonitorIDi: str):
    """
    Plot daily MDA8 O3 for one monitor with its AQS code in the title.
    
    Parameters
    ----------
    Monitori_MDA8O3 : xr.DataArray
        Daily O3 values for one site (dims: time)
    MonitorIDi : str
        Informative label for the site, e.g., its AQS code
    """
    fig, ax = plt.subplots(figsize=(8, 4))

    # Convert from mol/mol to ppb
    vals_ppb = Monitori_MDA8O3 * 1e9

    vals_ppb.plot(ax=ax, marker="o", linestyle="-", color="tab:blue", markersize=3)

    ax.set_ylabel("MDA8 O$_3$ (ppb)", fontsize=15)
    ax.set_xlabel("Date", fontsize=12)
    ax.set_title(f"MDA8 O$_3$ at Monitor {MonitorIDi}", fontsize=18)
    
    # Add horizontal line at 70 ppb
    ax.axhline(70, color="red", linestyle="--", linewidth=1)
    ax.text(
        x=vals_ppb["time"].values[len(vals_ppb) // 20],  # put near the left edge
        y=72, 
        s="MDA8 O$_3$ = 70 ppb",
        color="red",
        fontsize=15,
        va="bottom",
    )

    plt.tight_layout()
    plt.show()



In [ ]:
# Read in saved ones
casename = BASE2022_casename
YYYY = casename_label_dic[casename][-4:]
startfileDate = f'{YYYY}-04-01'
endfileDate = f'{YYYY}-10-31'


In [ ]:
get_site_index?

In [ ]:
allMonitors_MDA8O3_filename = (
        f"{Output_diri}ForGivenMonitors/LocalTimeMDA8O3.{casename_label_dic[casename]}."
        f"GivenMonitors.{startfileDate}T{endfileDate}.nc"
    )

allMonitors_HourlyO3_filename = (
        f"{Output_diri}ForGivenMonitors/UTChourlyO3.{casename_label_dic[casename]}."
        f"GivenMonitors.{startfileDate}T{endfileDate}.nc"
    )

print_file_size(allMonitors_MDA8O3_filename)
print_file_size(allMonitors_HourlyO3_filename)

In [ ]:
allMonitors_MDA8O3_ds = xr.open_dataset(allMonitors_MDA8O3_filename)
allMonitors_MDA8O3_ds

In [ ]:
allMonitors_MDA8O3_ds['MDA8O3'].values[1]

In [ ]:
# Datei = '2022-05-04'
# allMonitors_MDA8O3_ds.sel(time=f'{Datei}T00:00:00.000000000')['MDA8O3']

In [ ]:
allMonitors_HourlyO3_ds = xr.open_dataset(allMonitors_HourlyO3_filename)
allMonitors_HourlyO3_ds

In [ ]:
allMonitors_HourlyO3_ds['O3'].values[1]

In [ ]:
# Example call
Monitori_MDA8O3 = allMonitors_MDA8O3_ds['MDA8O3'].isel(AQS_code=200)
MonitorIDi = str(Monitori_MDA8O3.AQS_code.values)
plot_monitor_timeseries(Monitori_MDA8O3, MonitorIDi)  # replace with your site’s AQS code


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def map_monitors_MDA8O3_Datei(allMonitors_MDA8O3_ds, Datei,
                               vmin=20, vmax=80, markersize=20,
                               figsize=(9,6), cmap="viridis",
                               pad_deg=1.5):
    """
    Plot all-monitor MDA8 O3 for a given date on a map.

    Parameters
    ----------
    allMonitors_MDA8O3_ds : xr.Dataset
        Must contain:
          - data var 'MDA8O3' with dims ('AQS_code','time'), units ppb
          - coords 'lat'('AQS_code'), 'lon'('AQS_code')
    Datei : str
        Date string like 'YYYY-MM-DD' (e.g., '2022-05-04').
    vmin, vmax : float
        Color scale limits (ppb).
    markersize : int
        Dot size for monitors.
    figsize : tuple
        Figure size.
    cmap : str
        Matplotlib colormap name.
    pad_deg : float
        Degrees of padding added around data extent.
    """

    # Robust date selection (accepts 'YYYY-MM-DD' or full timestamp present in ds)
    target_time = pd.to_datetime(Datei)
    try:
        da = allMonitors_MDA8O3_ds["MDA8O3"].sel(time=target_time)*1e9
    except Exception:
        # If exact match fails, match by date (floor)
        times = pd.DatetimeIndex(allMonitors_MDA8O3_ds["time"].values)
        mask_time = times.normalize() == target_time.normalize()
        if not mask_time.any():
            raise KeyError(f"No time matching {Datei} in dataset.")
        da = allMonitors_MDA8O3_ds["MDA8O3"].isel(time=np.where(mask_time)[0][0])

    # Align lat/lon to the same AQS_code order as 'da'
    aqs_idx = da["AQS_code"]
    lats = allMonitors_MDA8O3_ds["lat"].sel(AQS_code=aqs_idx).values
    lons = allMonitors_MDA8O3_ds["lon"].sel(AQS_code=aqs_idx).values
    vals = da.values

    # Mask NaNs
    good = np.isfinite(vals) & np.isfinite(lats) & np.isfinite(lons)
    if good.sum() == 0:
        raise ValueError(f"All values are NaN for {Datei}.")

    lats, lons, vals = lats[good], lons[good], vals[good]

    # Lazy import cartopy only when plotting
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature

    fig, ax = plt.subplots(figsize=figsize,subplot_kw={'projection': ccrs.PlateCarree()})

    # Focus on CONUS
    ax.set_extent([-125, -66.5, 24, 49], crs=ccrs.PlateCarree())

    # Base map features
    ax.add_feature(cfeature.COASTLINE, linewidth=0.6)
    ax.add_feature(cfeature.BORDERS, linewidth=0.4)
    try:
        ax.add_feature(cfeature.STATES.with_scale("50m"), linewidth=0.3)
    except Exception:
        pass

    # Scatter plot of monitors
    sc = ax.scatter(lons, lats, c=vals, s=markersize, cmap=cmap,
                    vmin=vmin, vmax=vmax, transform=ccrs.PlateCarree())

    # Colorbar
    cbar = plt.colorbar(sc, ax=ax, orientation="vertical", pad=0.02, shrink=0.6)
    cbar.set_label("MDA8 O$_3$ (ppb)",fontsize=16)

    # Title
    ax.set_title(f"MDA8 O$_3$ at Monitors | {pd.to_datetime(Datei).date()}  "
                 f"(N={len(vals)})", fontsize=16)

    plt.tight_layout()
    plt.show()


In [ ]:
# allMonitors_MDA8O3_ds['MDA8O3']#.values

In [ ]:
# Example: Datei = '2022-05-04'
map_monitors_MDA8O3_Datei(allMonitors_MDA8O3_ds, '2022-07-28')
